### **02_semantic_search.ipynb**
### **Multilingual semantic-similarity search**

* ##### 01 - Install packages
* ##### 02 - Import packages
* ##### 03 - Create Elasticsearch client
* ##### 04 - Download Multilingual Universal Sentence Encoder model
* ##### 05 - Multilingual semantic-similarity search

### 01 - Install packages

In [ ]:
!pip install elasticsearch==8.11.1 tensorflow==2.15.0 tensorflow-text==2.15.0 tensorflow-hub==0.15.0

### 02 - Import packages

In [ ]:
import tensorflow_text as tf_text
import tensorflow_hub  as tf_hub

from elasticsearch import Elasticsearch

import urllib3
urllib3.disable_warnings()

### 03 - Create Elasticsearch client

In [ ]:
es_host     = '<elasticsearch_host>'
es_username = '<elasticsearch_username>'
es_password = '<elasticsearch_password>'

In [ ]:
es = Elasticsearch(
    hosts        = es_host,
    basic_auth   = (es_username, es_password),
    verify_certs = False
)

In [ ]:
es.info()

### 04 - Download Multilingual Universal Sentence Encoder model

In [ ]:
model = tf_hub.load('https://www.kaggle.com/models/google/universal-sentence-encoder/frameworks/TensorFlow2/variations/multilingual-large/versions/2')

In [ ]:
model('Hello World, ML Elasticsearch!')[0].numpy()

### 05 - Multilingual semantic-similarity search

In [ ]:
bbc_news_index = 'bbc_news'

In [ ]:
def semantic_search(text):

    query = {
        'script_score' : {
            'query'  : { 'match_all' : {} },
            'script' : {
                'source' : "cosineSimilarity(params.text_embeddings, 'text_embeddings') + 1.0",
                'params' : { 'text_embeddings' : model(text)[0].numpy() }
            }
        }
    }

    result = es.search(index = bbc_news_index, query = query, size = 1)
    result = result['hits']['hits']

    if len(result) == 0:

        print('no results found...')
        return

    result = result[0]

    print(f"score : { result['_score'] }")
    print(f"label : { result['_source']['label_text'] }")
    print(f"text  : { result['_source']['text'] }")

In [ ]:
semantic_search('economic growth')

In [ ]:
semantic_search('crescimento econômico')

In [ ]:
semantic_search('crecimiento económico')